# 7.5B 实现：历史工单智能检索与知识发布本 notebook 是同名设计文档（md）的实现篇：md 讲业务痛点、背景、解决的问题与架构，这里放全部代码、实现细节和细节讲解。按 0–5 节顺序执行，即可完整走通这条链路：检索可用、增量生效、坏数据拦截、回滚可读。向量化默认用智谱 embedding-3（OpenAI 兼容接口）；在下面第一个代码单元填入你的 key，或设置环境变量 `EMBEDDING_API_KEY`。## 0 环境准备

In [1]:
%pip install -q pymilvus==3.0.2 pyarrow boto3

## 1 数据准备> 整体流程与预期见 md 7.5B.1；下面开始动手。依赖与 7.1 的 standalone 环境一致：Milvus `v3.0.2`、MinIO、`pymilvus 3.0.2`，外加 `requests` 调 embedding 接口。下面的代码做四件事：配置 embedding 模型（默认智谱 embedding-3；key 从环境变量 `EMBEDDING_API_KEY` 读取，也可以直接填在 `api_key` 变量里）、连接 Milvus/MinIO、定义 80 条历史工单、把第一批批次向量化后写入湖。`endpoint_url` 两个视角别搞混：**Python 客户端访问 MinIO 用 `localhost:9000`，Milvus 容器访问 MinIO 用 `minio:9000`**。

In [2]:
import jsonimport osimport timefrom pathlib import Pathimport boto3import pyarrow as paimport pyarrow.parquet as pqimport requestsfrom botocore.config import Configfrom pymilvus import DataType, MilvusClient# ---- embedding 模型（OpenAI 兼容接口；默认智谱 embedding-3，1024 维）----EMBEDDING_BASE_URL = os.environ.get("EMBEDDING_BASE_URL", "https://open.bigmodel.cn/api/paas/v4")EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "embedding-3")EMBEDDING_DIM = int(os.environ.get("EMBEDDING_DIM", "1024"))api_key = os.environ.get("EMBEDDING_API_KEY", "在这里填入你的 API Key")assert "填" not in api_key, "请先填入 embedding 模型的 API Key（环境变量 EMBEDDING_API_KEY 或直接改这一行）"def embed(texts):    """分批调用；单批 32 条，避免超出服务商的批量上限。"""    vectors = []    for i in range(0, len(texts), 32):        r = requests.post(            f"{EMBEDDING_BASE_URL}/embeddings",            headers={"Authorization": f"Bearer {api_key}"},            json={"model": EMBEDDING_MODEL, "input": texts[i:i + 32],                  "dimensions": EMBEDDING_DIM},            timeout=60,        )        r.raise_for_status()        data = sorted(r.json()["data"], key=lambda d: d["index"])        vectors.extend(d["embedding"] for d in data)    return vectorsBUCKET = "city-kb-demo"PREFIX = "city-kb/releases"EXT_SOURCE = f"s3://{BUCKET}/{PREFIX}/"client = MilvusClient(uri="http://127.0.0.1:19530")print("Milvus:", client.get_server_version())s3 = boto3.client("s3", endpoint_url="http://localhost:9000",                  aws_access_key_id="minioadmin", aws_secret_access_key="minioadmin",                  config=Config(signature_version="s3v4"), region_name="us-east-1")try:    s3.head_bucket(Bucket=BUCKET)except Exception:    s3.create_bucket(Bucket=BUCKET)def complaint(complaint_id, district, category, title, description, handling, result):    return {        "complaint_id": complaint_id, "district": district, "category": category,        "title": title, "description": description,        "handling": handling, "result": result,    }PARQUET_SCHEMA = pa.schema([    ("complaint_id", pa.int64()),    ("district", pa.string()), ("category", pa.string()),    ("title", pa.string()), ("description", pa.string()),    ("handling", pa.string()), ("result", pa.string()),    ("embedding", pa.list_(pa.float32(), EMBEDDING_DIM)),])def upload_batch(rows, release_id):    """向量化一个批次并写成独立文件；同名路径不允许覆盖。"""    vectors = embed([r["description"] for r in rows])    rows = [dict(r, embedding=v) for r, v in zip(rows, vectors)]    table = pa.Table.from_pylist(rows, schema=PARQUET_SCHEMA)    local = Path(f"complaints_{release_id}.parquet")    pq.write_table(table, local, compression="snappy")    key = f"{PREFIX}/{release_id}.parquet"    same = s3.list_objects_v2(Bucket=BUCKET, Prefix=key).get("Contents", [])    if any(o["Key"] == key for o in same):        raise RuntimeError(f"batch key already exists: s3://{BUCKET}/{key}")    s3.upload_file(str(local), BUCKET, key)    print("batch:", f"s3://{BUCKET}/{key}", "rows:", table.num_rows)BASE_ROWS = [    complaint(        2001, "米市巷街道", "环境污染",        "米市巷农贸市场周边油烟直排",        "餐馆油烟未经净化直排，周边居民不敢开窗",        "责令加装油烟净化设备，并纳入每月巡查",        "已整改，安装净化设备两台",    ),    complaint(        2002, "上塘街道", "环境污染",        "上塘河沿街夜市噪声扰民",        "夜间排档喧哗到凌晨，多次投诉无改善",        "联合城管划定经营区域，限定经营时间",        "夜间噪声投诉明显下降",    ),    complaint(        2003, "朝晖街道", "市政设施",        "朝晖小区北门道路破损",        "路面坑洼积水，老人出行容易绊倒",        "列入市政维修计划，完成路面修复并同步改造排水",        "三天内完成修复",    ),    complaint(        2004, "东新街道", "市政设施",        "东新路人行道井盖破损",        "井盖错位破损，存在安全隐患",        "当日更换井盖并加固周边基座",        "已更换完毕",    ),    complaint(        2005, "米市巷街道", "物业管理",        "老小区电梯频繁停运",        "电梯多次困人，物业维修不及时",        "督促物业启用维修资金更换曳引机，并明确维保周期",        "电梯恢复正常运行",    ),    complaint(        2006, "朝晖街道", "物业管理",        "小区绿化带被改成停车位",        "物业擅自铲除绿化改设车位，业主不满",        "约谈物业责令恢复绿化，并组织业主议事会协商",        "绿化已恢复",    ),    complaint(        2007, "祥符街道", "违建占道",        "祥符街沿街商铺占道经营",        "商铺外摆占用消防通道，早晚高峰通行困难",        "联合执法清理占道物品，并划定临时外摆线",        "通道恢复畅通",    ),    complaint(        2008, "祥符街道", "违法建设",        "小区楼顶违规搭建阳光房",        "顶楼住户未审批搭建构筑物",        "立案查处，限期自行拆除并组织复核",        "已拆除",    ),]# 其余 72 条：覆盖 9 个街道 × 8 个类别，模拟真实湖表里大量与道路破损无关的工单。_DISTRICTS = ["米市巷街道", "上塘街道", "东新街道", "祥符街道", "东湖街道",              "西湖街道", "江城街道", "钱塘街道", "滨江街道"]_LOCS = ["朝阳", "锦园", "锦绣", "枫林", "春波", "望江", "文华", "金沙", "绿城"]_TEMPLATES = {    "环境污染": ("河道异味反复", "河边排口散发异味，居民投诉多次", "沿线排查封堵违规排口并清淤", "异味消除"),    "物业管理": ("二次供水压力不足", "小区高层高峰时段水压不足", "协调物业更换加压泵", "供水恢复稳定"),    "违建占道": ("沿街店铺占道堆物", "商铺门前堆放货物占用盲道", "清理占道物品并纳入网格巡查", "盲道恢复通行"),    "违法建设": ("楼顶违规加建彩钢棚", "顶楼住户未审批搭建构筑物", "立案调查并限期整改", "已拆除"),    "劳动保障": ("装修工人工资被拖欠", "小区装修项目拖欠三名工人工资两个月", "约谈包工头并督促支付", "工资已结清"),    "市场监管": ("预付卡退费纠纷", "门店停业后预付卡无法退款", "联合市场监管登记调解", "已退款"),    "教育咨询": ("学区范围划分咨询", "对新划入的学区范围有疑问", "核实政策依据并当面解释", "咨询人表示理解"),    "环境卫生": ("垃圾房清运不及时", "垃圾房满溢异味大，蚊虫滋生", "调整清运频次并增加消杀", "环境恢复整洁"),}for _di, _d in enumerate(_DISTRICTS):    for _ci, (_cat, (_t, _desc, _h, _r)) in enumerate(_TEMPLATES.items()):        BASE_ROWS.append(complaint(            3001 + _di * 8 + _ci, _d, _cat,            f"{_LOCS[_di]}小区{_t}",            f"{_desc}，位于{_d}",            _h, _r,        ))assert len(BASE_ROWS) == 80upload_batch(BASE_ROWS, "base_20261110")

Milvus: 3.0.2
batch: s3://city-kb-demo/city-kb/releases/base_20261110.parquet rows: 80


每次上传都写成新文件，文件名带发布号，已发布的文件不改写；刷新时把前缀下当前可见的文件重新解析成清单。## 2 查询面实现> 查询面设计见 md 7.5B.2。### 建外部 Collection外部 Collection 的 Schema 与普通 Collection 有两处不同：**不要声明用户主键**，并且每个字段通过 `external_field` 映射到 Parquet 列。`endpoint_url` 这里写 `minio:9000`，因为刷新作业由 Milvus 容器执行，不是由本机 Python 执行。

In [3]:
EXT_SPEC = json.dumps({    "format": "parquet",    "extfs": {        "access_key_id": "minioadmin",        "access_key_value": "minioadmin",        "region": "us-east-1",        "cloud_provider": "aws",        "endpoint_url": "http://minio:9000",        "use_ssl": "false",        "use_virtual_host": "false",    },})if client.has_collection("complaints"):    raise RuntimeError("complaints 已存在；先清理 7.5B 演示资源，避免误覆盖")schema = MilvusClient.create_schema(    external_source=EXT_SOURCE,    external_spec=EXT_SPEC,)schema.add_field(    field_name="complaint_id", datatype=DataType.INT64, external_field="complaint_id")schema.add_field(    field_name="district", datatype=DataType.VARCHAR, max_length=128,    external_field="district",)schema.add_field(    field_name="category", datatype=DataType.VARCHAR, max_length=16,    external_field="category",)schema.add_field(    field_name="title", datatype=DataType.VARCHAR, max_length=512,    external_field="title",)schema.add_field(    field_name="description", datatype=DataType.VARCHAR, max_length=2048,    external_field="description",)schema.add_field(    field_name="handling", datatype=DataType.VARCHAR, max_length=2048,    external_field="handling",)schema.add_field(    field_name="result", datatype=DataType.VARCHAR, max_length=1024,    external_field="result",)schema.add_field(    field_name="embedding", datatype=DataType.FLOAT_VECTOR, dim=EMBEDDING_DIM,    external_field="embedding",)client.create_collection(collection_name="complaints", schema=schema)index_params = client.prepare_index_params()for field in ["complaint_id", "district", "category"]:    index_params.add_index(field_name=field, index_type="AUTOINDEX")index_params.add_index(    field_name="embedding", index_type="AUTOINDEX", metric_type="COSINE")client.create_index(collection_name="complaints", index_params=index_params)

### 刷新不是「再查一次」，而是重建视图外部 Collection 建好时，Milvus 只知道源前缀，不知道当前有哪些文件。`refresh_external_collection` 会生成新一版数据集清单；它异步执行，返回 `job_id`。刷新完成后必须让查询节点重新加载，否则可能还读着旧清单。

In [4]:
def refresh_and_wait(collection, timeout=180):    job_id = client.refresh_external_collection(collection_name=collection)    deadline = time.time() + timeout    job = None    while time.time() < deadline:        jobs = client.list_refresh_external_collection_jobs(            collection_name=collection        )        job = next(            (j for j in jobs if str(j.job_id) == str(job_id)),            None,        )        if job is None:            time.sleep(1)            continue        state = str(job.state)        if "Completed" in state:            break        if "Failed" in state:            raise RuntimeError(f"refresh failed: {getattr(job, 'reason', state)}")        time.sleep(1)    else:        raise TimeoutError(f"refresh timeout: {collection}, job={job_id}")    # 7.2.6 已说明：刷新完成后要重新加载，查询节点才会使用新清单。    try:        client.release_collection(collection_name=collection)    except Exception:        pass    client.load_collection(collection_name=collection)    return job_idrefresh_and_wait("complaints")

### 检索建议接口建议检索不是“拿诉求文本直接全库乱搜”。先按区域与类别缩小候选，再在候选内按向量距离排序。这样能避免“道路破损”这种通用词把其他区域的工单排到前面。md 7.5B.2 图里的 FunctionChain 是可选的排序扩展：需要按业务规则加权（例如让“已解决”的工单靠前）时，在同一次查询里声明排序规则即可。本实现只用「过滤 + 相似度排序」，取前 3 条返回。

In [5]:
def advise(query_text, district, category, top_k=3, collection=None):    collection = collection or STATE["active_collection"]    filter_expr = (        f"district == {json.dumps(district)} && "        f"category == {json.dumps(category)}"    )    qvec = embed([query_text])[0]    result = client.search(        collection_name=collection,        data=[qvec],        anns_field="embedding",        search_params={"metric_type": "COSINE"},        filter=filter_expr,        limit=top_k,        output_fields=[            "complaint_id", "district", "category",            "title", "description", "handling",        ],    )    hits = []    for hit in result[0]:        entity = hit["entity"]        hits.append({            "complaint_id": entity["complaint_id"],            "district": entity["district"],            "category": entity["category"],            "title": entity["title"],            "description": entity["description"],            "handling": entity["handling"],            "score": float(hit["distance"]),        })    return hitsdef collection_count(collection):    rows = client.query(        collection_name=collection,        filter="",        output_fields=["count(*)"],    )    return int(rows[0]["count(*)"])

注意：`STATE` 要到发布控制面才定义。为了让查询代码可以先独立运行，这里先写一个最小状态文件；后面新增、好批次和坏批次都读写同一份指针。

In [6]:
STATE_FILE = Path("complaint_release_state.json")if STATE_FILE.exists():    STATE = json.loads(STATE_FILE.read_text(encoding="utf-8"))else:    STATE = {        "active_collection": "complaints",        "release": "base_20261110",        "rollback_checkpoint": None,    }def save_state(**changes):    # 单机演示用临时文件 + 原子替换；生产应放进数据库或配置中心，并加版本号。    new_state = {**STATE, **changes}    tmp = STATE_FILE.with_suffix(".tmp")    tmp.write_text(        json.dumps(new_state, ensure_ascii=False, indent=2),        encoding="utf-8",    )    tmp.replace(STATE_FILE)    STATE.clear()    STATE.update(new_state)

### 四条高频问题先对齐预期高频问题搜索不是为了“展示搜索结果”，而是发布门禁的判定依据。每条都固定了区域、类别、期望工单号和处置关键词。

In [7]:
ROAD_QUERY = "朝晖小区北门路面破损，坑洼积水，老人出行困难"GOLDEN_CASES = [    {        "name": "road_damage",        "query": ROAD_QUERY,        "district": "朝晖街道",        "category": "市政设施",        "expected_id": 2003,        "keyword": "路面修复",    },    {        "name": "oil_smoke",        "query": "小区楼下餐馆油烟直排，窗户不敢开，夜里味道更大",        "district": "米市巷街道",        "category": "环境污染",        "expected_id": 2001,        "keyword": "油烟净化",    },    {        "name": "property_elevator",        "query": "老小区电梯频繁困人停运，物业维修拖了很久",        "district": "米市巷街道",        "category": "物业管理",        "expected_id": 2005,        "keyword": "维保",    },    {        "name": "illegal_occupation",        "query": "沿街店铺占道经营，人行道都走不通",        "district": "祥符街道",        "category": "违建占道",        "expected_id": 2007,        "keyword": "占道",    },]GOOD_CASE = {    "name": "night_construction",    "query": "燃气管道施工开挖后路面沉降",    "district": "朝晖街道",    "category": "市政设施",    "expected_id": 2101,    "keyword": "回填碾压",}assert collection_count("complaints") == len(BASE_ROWS) == 80top1 = advise(ROAD_QUERY, "朝晖街道", "市政设施", top_k=1)[0]assert top1["complaint_id"] == 2003assert "路面修复" in top1["handling"]

到这里，系统已经能回答“这件新诉求最像哪件历史工单”。但它还只是一个查询服务；真正的端到端实战在于**下一批工单能不能安全地进入这个视图**。---## 3 发布面实现> 发布面设计见 md 7.5B.3。### 门禁不是看接口通不通，而是看业务结果对不对每次刷新后，系统要回答两个问题：1. 数据是不是完整进入了当前清单？2. 高频问题搜索的最相似工单和处置关键词，是不是仍然正确？

In [8]:
def run_gate(collection, expected_count, cases):    failures = []    actual_count = collection_count(collection)    if actual_count != expected_count:        failures.append(            f"row_count: expected {expected_count}, got {actual_count}"        )    for case in cases:        hits = advise(            case["query"],            case["district"],            case["category"],            top_k=1,            collection=collection,        )        if not hits:            failures.append(f"{case['name']}: no hit")            continue        top = hits[0]        if top["complaint_id"] != case["expected_id"]:            failures.append(                f"{case['name']}: expected {case['expected_id']}, "                f"got {top['complaint_id']} ({top['title']})"            )        if case["keyword"] not in top["handling"]:            failures.append(                f"{case['name']}: handling missing "                f"{case['keyword']!r}: {top['handling']!r}"            )    return {        "passed": not failures,        "expected_count": expected_count,        "actual_count": actual_count,        "failures": failures,    }

门禁只在**候选状态**上判定；通过之前，不更新 `active_collection`。注意这里仍是原地刷新：刷新完成到门禁判完之间，当前集合已经能看到新批次，生产要消除这个窗口，应改成另建候选集合刷新、门禁通过再切指针（见 md 7.5B.6）。失败时不能只打印一条错误日志——当前外部集合已经刷进了坏批次，必须立刻恢复发布前快照，否则业务读的仍是坏视图。### 快照恢复是异步的，而且产出新集合外部 Collection 的快照记录的是当时的清单，不复制湖上的 Parquet。恢复快照会创建一个**新 Collection**，原集合一行不动；应用必须自己把读指针切过去。这个“切读”动作是系统的一部分，不是一句运维备注。

In [9]:
def restore_and_wait(snapshot_name, source_collection, target_collection,                     timeout=600):    if client.has_collection(target_collection):        raise RuntimeError(f"restore target already exists: {target_collection}")    job_id = client.restore_snapshot(        snapshot_name=snapshot_name,        source_collection_name=source_collection,        target_collection_name=target_collection,    )    deadline = time.time() + timeout    while time.time() < deadline:        info = client.get_restore_snapshot_state(job_id=job_id)        state = str(info.state)        raw_progress = getattr(info, "progress", 0)        progress = (            0 if raw_progress in (None, "")            else int(float(str(raw_progress).strip("%")))        )        print(f"restore {job_id}: {state}, {progress}%")        if "Failed" in state:            raise RuntimeError(f"restore failed: {getattr(info, 'reason', state)}")        if "Completed" in state:            break        time.sleep(1)    else:        raise TimeoutError(f"restore timeout: {snapshot_name}")    client.load_collection(collection_name=target_collection)    return target_collection

### 发布控制器下面的 `publish_batch` 是最小但完整的发布编排。它不是数据库意义上的原子事务；每一步的外部副作用，都要由状态机、门禁和恢复动作处理：1. 记录当前行数；2. 对当前活动集合拍预发布快照；3. 把新批次写入湖；4. 刷新并重新加载；5. 跑验收门禁；6. 通过则更新发布状态；失败则恢复快照、在恢复集合上重跑“上一版”的门禁，最后才切换读指针。

In [10]:
def publish_batch(rows, release_id, acceptance_cases, rollback_cases):    active = STATE["active_collection"]    before_count = collection_count(active)    snapshot_name = f"snap_{release_id}"    if snapshot_name in client.list_snapshots(collection_name=active):        raise RuntimeError(f"snapshot already exists: {snapshot_name}")    client.create_snapshot(        snapshot_name=snapshot_name,        collection_name=active,        description=f"pre-release checkpoint for {release_id}",    )    upload_batch(rows, release_id)    refresh_and_wait(active)    candidate_gate = run_gate(        collection=active,        expected_count=before_count + len(rows),        cases=acceptance_cases,    )    if candidate_gate["passed"]:        save_state(            active_collection=active,            release=release_id,            rollback_checkpoint=snapshot_name,        )        return {            "status": "published",            "release_id": release_id,            "active_collection": active,            "gate": candidate_gate,            "snapshot": snapshot_name,        }    # 门禁失败：当前活动集合已经被刷新，不能再让业务继续读它。    rollback_collection = f"complaints_rb_{release_id}"    restore_and_wait(        snapshot_name=snapshot_name,        source_collection=active,        target_collection=rollback_collection,    )    rollback_gate = run_gate(        collection=rollback_collection,        expected_count=before_count,        cases=rollback_cases,    )    if not rollback_gate["passed"]:        raise RuntimeError(            f"rollback gate failed, active stays {active}: "            f"{rollback_gate['failures']}"        )    save_state(        active_collection=rollback_collection,        release=f"rollback_{release_id}",        rollback_checkpoint=snapshot_name,    )    return {        "status": "rolled_back",        "release_id": release_id,        "active_collection": rollback_collection,        "failed_gate": candidate_gate,        "rollback_gate": rollback_gate,        "snapshot": snapshot_name,    }

这里有三个容易被忽略的工程点：- **先拍快照，再上传批次**。快照要代表“坏批次出现之前”的版本，顺序反了就失去恢复点。- **恢复集合也必须过门禁**。恢复成功不等于数据正确；恢复出的集合若行数不对、索引未加载，仍然不能切读。- **切指针是发布的一部分**。`restore_snapshot` 不会修改原集合，也不会自动改变业务查询地址。### 什么叫“发布成功”发布不是“`refresh` 返回了 job_id”，而是下面三个条件同时满足：

```textcandidate_gate.passed == TrueAND active_collection 指向已通过门禁的集合AND 业务查询实际读取该集合```

如果只有前两步，应用还在查旧地址，发布就是假的；如果只有 API 成功，门禁没跑，坏数据照样会进入用户视野。---## 4 正常增量运行> 本节设计见 md 7.5B.4。### 新批次一个月后，朝晖小区北门又出了一单，但这次成因是夜间燃气管道施工后回填不实，路面反复沉降。处置已办结，热线平台把它导出成一条新记录：

In [11]:
GOOD_ROWS = [    complaint(        2101, "朝晖街道", "市政设施",        "朝晖路燃气管道开挖段路面沉降",        "夜间施工回填不实，路面多次沉降",        "责成施工单位重新回填碾压，并列入季度养护清单",        "复验合格",    ),]

这条新工单与历史工单 `WO-2003` 同属“市政设施”问题族，但语义重点略有偏移，代表问题重点从「路面破损」转到「回填沉降」。发布时要同时验证两件事：- 新诉求应召回 `WO-2101`；- 原有的道路破损诉求不能被新记录挤掉，仍应召回 `WO-2003`。### 通过门禁并发布

In [12]:
good_result = publish_batch(    rows=GOOD_ROWS,    release_id="20261110_good",    acceptance_cases=GOLDEN_CASES + [GOOD_CASE],    rollback_cases=GOLDEN_CASES,)print(json.dumps(good_result, ensure_ascii=False, indent=2))assert good_result["status"] == "published"assert good_result["gate"]["passed"] is Trueassert good_result["gate"]["actual_count"] == 81assert good_result["active_collection"] == "complaints"assert STATE["release"] == "20261110_good"

batch: s3://city-kb-demo/city-kb/releases/20261110_good.parquet rows: 1
{
  "status": "published",
  "release_id": "20261110_good",
  "active_collection": "complaints",
  "gate": {
    "passed": true,
    "expected_count": 81,
    "actual_count": 81,
    "failures": []
  },
  "snapshot": "snap_20261110_good"
}


发布成功后：行数 80 → 81，`WO-2101` 可查，`active_collection` 仍是 `complaints`（内容已是新版本）；快照 `snap_20261110_good` 是发布**之前**的恢复点（80 行基线）。此时再问系统一次：

In [13]:
night_top1 = advise(    "朝晖路燃气管道开挖段路面反复沉降，夜间施工回填不实",    "朝晖街道", "市政设施", top_k=1)[0]assert night_top1["complaint_id"] == 2101assert "回填碾压" in night_top1["handling"]road_top1 = advise(    ROAD_QUERY, "朝晖街道", "市政设施", top_k=1)[0]assert road_top1["complaint_id"] == 2003assert "路面修复" in road_top1["handling"]print("夜间施工诉求 Top-1:", night_top1["title"], "|", night_top1["handling"])print("道路破损诉求 Top-1:", road_top1["title"], "|", road_top1["handling"])

夜间施工诉求 Top-1: 朝晖路燃气管道开挖段路面沉降 | 责成施工单位重新回填碾压，并列入季度养护清单
道路破损诉求 Top-1: 朝晖小区北门道路破损 | 列入市政维修计划，完成路面修复并同步改造排水


新知识可查，旧知识没有被覆盖，这才是一次有业务意义的增量。---## 5 坏增量与回滚> 本节设计见 md 7.5B.5。### 坏数据长什么样第二天的批次里混进了一条错误工单：它和高频道路问题的诉求文本完全相同，但处置只写了“该诉求已办结归档”，没有任何实际处置动作。原因可能是街道集中清理积压来电时，把重复诉求错误标记成了已办结。无论原因是什么，如果直接刷新上线，工作人员会得到“问题已解决”的错误参考。

In [14]:
BAD_ROWS = [    complaint(        9901, "朝晖街道", "市政设施",        "朝晖小区北门道路破损（重复来电）",        "朝晖小区北门路面破损，坑洼积水，老人出行困难",        "该诉求已办结归档",        "已办结",    ),]

这条记录的危险性不在“多了一行”，而在于它和高频道路问题的诉求文本完全相同：- 对“道路破损”诉求，`WO-9901` 会压过真正正确的 `WO-2003`；- 它的处置却是“已办结归档”，没有任何可执行动作，会让工作人员误以为问题已解决；- 只检查 `count(*)` 从 81 变成 82，完全发现不了这个问题。### 发布门禁按预期失败

In [15]:
bad_result = publish_batch(    rows=BAD_ROWS,    release_id="20261110_bad",    acceptance_cases=GOLDEN_CASES + [GOOD_CASE],    rollback_cases=GOLDEN_CASES + [GOOD_CASE],)print(json.dumps(bad_result, ensure_ascii=False, indent=2))

batch: s3://city-kb-demo/city-kb/releases/20261110_bad.parquet rows: 1
restore 470381299908194578: RestoreSnapshotPending, 0%
restore 470381299908194578: RestoreSnapshotPending, 0%
restore 470381299908194578: RestoreSnapshotExecuting, 0%
restore 470381299908194578: RestoreSnapshotExecuting, 0%
restore 470381299908194578: RestoreSnapshotExecuting, 0%
restore 470381299908194578: RestoreSnapshotExecuting, 0%
restore 470381299908194578: RestoreSnapshotCompleted, 100%
{
  "status": "rolled_back",
  "release_id": "20261110_bad",
  "active_collection": "complaints_rb_20261110_bad",
  "failed_gate": {
    "passed": false,
    "expected_count": 82,
    "actual_count": 82,
    "failures": [
      "road_damage: expected 2003, got 9901 (朝晖小区北门道路破损（重复来电）)",
      "road_damage: handling missing '路面修复': '该诉求已办结归档'"
    ]
  },
  "rollback_gate": {
    "passed": true,
    "expected_count": 81,
    "actual_count": 81,
    "failures": []
  },
  "snapshot": "snap_20261110_bad"
}


执行逻辑要避免“坏批次先被用户看到，再等人工发现”，具体步骤是：1. 对当前 `complaints` 拍 `snap_20261110_bad`；2. 上传坏批次并刷新；3. 门禁发现道路用例的 Top-1 变成了 `WO-9901`；4. 调用 `restore_snapshot`，恢复成新集合 `complaints_rb_20261110_bad`；5. 在恢复集合上重新跑 81 行版本的全部高频问题搜索；6. 恢复门禁通过后，把 `active_collection` 切到恢复集合；7. 返回 `status=rolled_back`，原 `complaints` 不再作为业务读入口。恢复后的系统状态应当满足：

In [16]:
assert bad_result["status"] == "rolled_back"assert bad_result["failed_gate"]["passed"] is Falseassert any(    "road_damage" in failure    for failure in bad_result["failed_gate"]["failures"])assert bad_result["rollback_gate"]["passed"] is Trueassert STATE["active_collection"] == "complaints_rb_20261110_bad"assert collection_count(STATE["active_collection"]) == 81

注意最后两个断言：回滚不是“恢复作业成功”就结束，而是**恢复集合行数正确、门禁通过、业务指针已经切过去**。三者缺一，事故仍在。### 从业务入口再验一次回滚之后，业务不该关心底层集合叫什么，只需通过 `active_collection` 查询：

In [17]:
final_top1 = advise(    ROAD_QUERY, "朝晖街道", "市政设施", top_k=1)[0]assert final_top1["complaint_id"] == 2003assert "路面修复" in final_top1["handling"]# 恢复视图里不应再出现被污染的工单 9901。all_titles = [    row["title"]    for row in advise(ROAD_QUERY, "朝晖街道", "市政设施", top_k=10)]assert "朝晖小区北门道路破损（重复来电）" not in all_titles

最终业务看到的是：- 道路破损诉求 → `WO-2003` → 「列入市政维修计划，完成路面修复并同步改造排水」；- 坏批次 `WO-9901` 已被隔离在恢复视图之外；- 读入口是 `complaints_rb_20261110_bad`，不是已经被污染的 `complaints`。

```text业务结果：  输入：朝晖小区北门道路破损诉求  Top-1：WO-2003 朝晖小区北门道路破损  处置：列入市政维修计划，完成路面修复并同步改造排水```

### 回滚之后，坏文件还在湖里这是系统必须讲清楚的一步。快照恢复只恢复 Milvus 的读视图，**不会删除对象存储中的坏 Parquet**；`complaints` 这个外部集合也仍然指向包含坏文件的前缀。此时如果谁把读指针切回 `complaints`，坏数据会重新出现。正确处置顺序是：1. 保持业务读取恢复集合；2. 数据治理侧把坏批次移出 `city-kb/releases/` 源前缀，或标记为不可见；3. 需要继续增量时，从干净前缀重建一个新的外部 Collection，过同一套门禁；4. 新集合适配通过后，再把 `active_collection` 从恢复集合切到新集合；5. 旧恢复集合保留一个回滚窗口，确认无异常后再删除。这段流程也解释了为什么“快照回滚”不是备份：它提供的是一个快速、可验证的读版本，不负责修复上游数据，更不能阻止下一次错误批次再次落湖。---全部单元跑通，即完成整条链路的验证：检索可用、增量生效、坏数据拦截、回滚可读。